# Stage 9+10 — QLoRA Fine-tuning: Mistral-7B on Security Threat Reports

**Goal:** Fine-tune `mistralai/Mistral-7B-Instruct-v0.3` on synthetic (Stage 9 input → structured JSON report) pairs.

**Output:** LoRA adapter pushed to `sohomn/stage9-10-explain-report`
- `adapter_model.safetensors` (~40MB)
- `adapter_config.json`
- `tokenizer.json` + config files

**Platform:** Kaggle T4 x1 (free tier sufficient)

**Training config:**
- QLoRA: 4-bit NF4, rank=16, alpha=32, dropout=0.05
- Target modules: q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj
- ~500 synthetic training pairs generated from Trinetra pipeline schema
- ~3-4h on T4

**Pipeline alignment:**
- Input: Stage 9 JSON (shap, attack_path, mitre_tactics, severity, threat_score)
- Output: structured JSON with summary, attack_narrative, remediation_steps[3], estimated_blast_radius
- Same schema as `REPORT_SCHEMA` in `app.py`

In [ ]:
# ── CELL 1: INSTALL ──────────────────────────────────────────────────────────
import subprocess, sys

def _pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + list(args), check=True)

_pip(
    'transformers>=4.43.0',
    'peft>=0.11.1',
    'bitsandbytes>=0.43.1',
    'accelerate>=0.30.0',
    'datasets>=2.19.0',
    'huggingface_hub>=0.23.0',
    'torch>=2.0.0',
)
print('✓ All packages installed')

In [ ]:
# ── CELL 2: CONFIG + AUTH ────────────────────────────────────────────────────
import os, json, random, re
import torch
from pathlib import Path
from huggingface_hub import login as hf_login

# ── HuggingFace auth ─────────────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

if not HF_TOKEN:
    raise ValueError('HF_TOKEN not found. Add it to Kaggle Secrets as HF_TOKEN.')
hf_login(token=HF_TOKEN, add_to_git_credential=False)
print('✓ HuggingFace authenticated')

# ── Paths + constants ────────────────────────────────────────────────────────
SEED          = 42
DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BASE_MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.3'
REPO_ID       = 'sohomn/stage9-10-explain-report'
OUT_DIR       = Path('/kaggle/working/stage9_10_finetune')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Feature vector slices — must match Stage 5 exactly ───────────────────────
Z_LOG_SLICE      = (0,   256)
Z_CVE_SLICE      = (256, 384)
RISK_SLICE       = (384, 385)
EXPLOIT_SLICE    = (385, 386)
Z_IDENTITY_SLICE = (386, 514)

# ── MITRE ATT&CK tactic pool ─────────────────────────────────────────────────
MITRE_TACTICS = [
    'TA0001 - Initial Access',
    'TA0002 - Execution',
    'TA0003 - Persistence',
    'TA0004 - Privilege Escalation',
    'TA0005 - Defense Evasion',
    'TA0006 - Credential Access',
    'TA0007 - Discovery',
    'TA0008 - Lateral Movement',
    'TA0009 - Collection',
    'TA0010 - Exfiltration',
    'TA0011 - Command and Control',
    'TA0040 - Impact',
    'TA0043 - Reconnaissance',
]

random.seed(SEED)
torch.manual_seed(SEED)
print(f'Device: {DEVICE}')
print(f'Base model: {BASE_MODEL_ID}')
print(f'Target repo: {REPO_ID}')

In [ ]:
# ── CELL 3: SYNTHETIC TRAINING DATA GENERATION ───────────────────────────────
# Generates 500 (Stage9_input, ideal_report) pairs grounded in the
# Trinetra pipeline schema — attack phases, cloud providers, node types,
# CVEs, MITRE tactics. Covers all severity tiers and node types.

NODE_TYPES     = ['User', 'VM', 'Role', 'Storage', 'Container', 'IP', 'CloudAccount']
PROVIDERS      = ['AWS', 'Azure', 'GCP']
ATTACK_PHASES  = ['privilege_escalation', 'lateral_movement', 'cve_exploitation',
                  'exfiltration', 'discovery', 'cross_cloud_pivot']
REL_TYPES      = ['ASSUMES_ROLE', 'ACCESS', 'CONNECTS_TO', 'EXPLOITS',
                  'HAS_VULNERABILITY', 'DEPLOYED_ON', 'CROSS_CLOUD_ACCESS']
REAL_CVES      = [
    'CVE-2021-44228',  # Log4Shell
    'CVE-2021-26855',  # ProxyLogon
    'CVE-2021-34527',  # PrintNightmare
    'CVE-2020-1472',   # Zerologon
    'CVE-2022-26134',  # Confluence RCE
    'CVE-2023-23397',  # Outlook NTLM
    'CVE-2023-44487',  # HTTP/2 Rapid Reset
    'CVE-2022-30190',  # Follina
    'CVE-2021-21985',  # VMware vCenter
    'CVE-2022-22965',  # Spring4Shell
]

DRIVER_LABELS = {
    'phi_log':      'anomalous log behaviour pattern',
    'phi_cve':      'high CVE exposure and exploit probability',
    'phi_risk':     'elevated CVSS risk score',
    'phi_exploit':  'active exploitation probability',
    'phi_identity': 'suspicious cross-cloud identity movement',
}


def _rand_shap(threat_score):
    """Generate realistic SHAP values that sum to threat_score."""
    raw = {k: random.random() for k in DRIVER_LABELS}
    total = sum(raw.values())
    return {k: round((v/total)*threat_score, 4) for k, v in raw.items()}


def _make_attack_path(node_id, phase):
    phase_paths = {
        'privilege_escalation': [
            f'{node_id} --[ASSUMES_ROLE]--> role_admin --[ACCESS]--> vm_prod',
            f'{node_id} --[ASSUMES_ROLE]--> role_cross_account --[CROSS_CLOUD_ACCESS]--> cloud_b',
        ],
        'lateral_movement': [
            f'{node_id} --[CONNECTS_TO]--> vm_internal --[ACCESS]--> storage_sensitive',
            f'{node_id} --[CROSS_CLOUD_ACCESS]--> azure_vm --[ASSUMES_ROLE]--> role_admin',
        ],
        'cve_exploitation': [
            f'{node_id} --[EXPLOITS]--> cve_node --[HAS_VULNERABILITY]--> vm_prod',
            f'{node_id} --[HAS_VULNERABILITY]--> service_endpoint --[ACCESS]--> db_prod',
        ],
        'exfiltration': [
            f'{node_id} --[ACCESS]--> storage_sensitive --[BELONGS_TO]--> cloud_account',
            f'{node_id} --[CONNECTS_TO]--> s3_bucket --[ACCESS]--> pii_data',
        ],
        'discovery': [
            f'{node_id} --[ACCESS]--> metadata_service --[BELONGS_TO]--> vpc',
        ],
        'cross_cloud_pivot': [
            f'{node_id} --[CROSS_CLOUD_ACCESS]--> gcp_vm --[ASSUMES_ROLE]--> gcp_admin',
            f'{node_id} --[CROSS_CLOUD_ACCESS]--> azure_ad --[ACCESS]--> azure_storage',
        ],
    }
    options = phase_paths.get(phase, [f'{node_id} --[ACCESS]--> target_resource'])
    return random.choice(options)


def _make_remediation(severity, node_type, cves):
    steps = {
        'Critical': [
            f'IMMEDIATELY isolate {node_type} from all network segments and revoke active sessions.',
            f'Initiate IR playbook P1. Notify SOC lead and CISO within 15 minutes.',
            f'Preserve forensic artifacts (memory dump, disk snapshot) before termination.'
        ],
        'High': [
            f'Quarantine {node_type} within 1 hour. Rotate all associated credentials immediately.',
            f'Alert SOC team. Begin root cause analysis. Review CloudTrail/Activity logs for past 24h.',
            f'Apply patches for {cves[0] if cves else "identified vulnerabilities"} within 4 hours.'
        ],
        'Medium': [
            f'Schedule {node_type} credential rotation within 24 hours.',
            f'Enable enhanced monitoring and alerting on this node for 72 hours.',
            f'Review access patterns and remove unnecessary permissions per least-privilege policy.'
        ],
        'Low': [
            f'Log event for periodic security review. No immediate action required.',
            f'Verify {node_type} configuration against security baseline at next scheduled audit.',
            f'Document in risk register for quarterly review.'
        ],
    }
    return steps.get(severity, steps['Medium'])


def _make_blast_radius(node_type, phase, threat_score):
    if threat_score >= 0.90:
        templates = {
            'User':         'Full account compromise likely. All resources accessible via assumed roles at risk. Cross-cloud propagation probable.',
            'VM':           'Host-level access exposes all co-located workloads, secrets in instance metadata, and attached storage volumes.',
            'Role':         'All principals that can assume this role are compromised. Audit all AssumeRole events in last 72h.',
            'Storage':      'All objects in bucket/container at risk of exfiltration. Check for public ACLs and cross-account access.',
            'Container':    'Container escape risk. Host OS and adjacent containers in same pod/task at risk.',
            'CloudAccount': 'Full account takeover. All child resources, cross-account trusts, and federated identities at risk.',
        }
    elif threat_score >= 0.75:
        templates = {
            'User':         'Lateral movement to connected VMs and roles possible. Credential reuse risk across cloud accounts.',
            'VM':           'Adjacent VMs in same VPC/subnet and shared EBS snapshots potentially exposed.',
            'Role':         'Resources accessible via this role at medium risk. Review trust policy immediately.',
            'Storage':      'Sensitive data in storage at risk. Verify encryption and access controls.',
            'Container':    'Application secrets and adjacent services in same namespace at risk.',
            'CloudAccount': 'Specific service endpoints and IAM principals at risk of compromise.',
        }
    else:
        templates = {
            'User':         'Limited blast radius. Risk contained to direct resource access of this principal.',
            'VM':           'Impact likely limited to this instance and directly attached resources.',
            'Role':         'Low-privilege role — limited impact if compromised.',
            'Storage':      'Non-sensitive storage — verify data classification before downgrading priority.',
            'Container':    'Isolated container — verify network policies prevent lateral movement.',
            'CloudAccount': 'Limited cross-account trust exposure.',
        }
    return templates.get(node_type, 'Impact scope undetermined — conduct manual assessment.')


def _make_summary(node_id, node_type, severity, phase, primary_driver, provider, threat_score):
    phase_desc = {
        'privilege_escalation': 'privilege escalation attempt',
        'lateral_movement':     'lateral movement across cloud resources',
        'cve_exploitation':     'active CVE exploitation',
        'exfiltration':         'data exfiltration activity',
        'discovery':            'reconnaissance and discovery operation',
        'cross_cloud_pivot':    'cross-cloud identity pivot',
    }.get(phase, 'anomalous activity')

    driver_desc = DRIVER_LABELS.get(primary_driver, 'multiple anomalous signals')

    return (
        f'{node_type} {node_id} on {provider} has been flagged with {severity} severity '
        f'(threat score: {threat_score:.2f}) due to {phase_desc}. '
        f'Primary detection signal: {driver_desc}. '
        f'Immediate investigation is {"required" if severity in ("Critical","High") else "recommended"}.'
    )


def _make_narrative(node_id, node_type, phase, attack_path, shap, cves, provider):
    phi_max = max(shap, key=shap.get)
    phase_narrative = {
        'privilege_escalation': (
            f'{node_type} {node_id} on {provider} performed unauthorized role assumption, '
            f'gaining elevated permissions beyond its baseline access profile. '
            f'Attack path: {attack_path}. '
            f'The {DRIVER_LABELS[phi_max]} ({shap[phi_max]:.4f}) was the strongest indicator.'
        ),
        'lateral_movement': (
            f'{node_type} {node_id} on {provider} initiated connections to internal resources '
            f'inconsistent with its normal access pattern, indicating lateral movement. '
            f'Movement path: {attack_path}. '
            f'Cross-cloud identity anomaly score: {shap.get("phi_identity", 0):.4f}.'
        ),
        'cve_exploitation': (
            f'{node_type} {node_id} on {provider} was involved in exploitation of '
            f'known vulnerabilities ({", ".join(cves[:2]) if cves else "unidentified CVEs"}). '
            f'Exploit chain: {attack_path}. '
            f'CVE risk contribution: {shap.get("phi_cve", 0):.4f}.'
        ),
        'exfiltration': (
            f'{node_type} {node_id} on {provider} accessed sensitive storage resources '
            f'at a volume and frequency inconsistent with baseline behaviour. '
            f'Data access path: {attack_path}. '
            f'Log anomaly contribution: {shap.get("phi_log", 0):.4f}.'
        ),
        'cross_cloud_pivot': (
            f'{node_type} {node_id} used federated credentials to pivot from {provider} '
            f'to resources in another cloud provider, bypassing single-cloud security controls. '
            f'Pivot path: {attack_path}. '
            f'Identity anomaly contribution: {shap.get("phi_identity", 0):.4f}.'
        ),
    }
    return phase_narrative.get(
        phase,
        f'{node_type} {node_id} on {provider} exhibited anomalous behaviour along path: {attack_path}.',
    )


def _make_mitre(phase, primary_driver):
    phase_tactics = {
        'privilege_escalation': ['TA0004 - Privilege Escalation', 'TA0001 - Initial Access'],
        'lateral_movement':     ['TA0008 - Lateral Movement',    'TA0004 - Privilege Escalation'],
        'cve_exploitation':     ['TA0002 - Execution',           'TA0006 - Credential Access'],
        'exfiltration':         ['TA0010 - Exfiltration',        'TA0009 - Collection'],
        'discovery':            ['TA0007 - Discovery',           'TA0043 - Reconnaissance'],
        'cross_cloud_pivot':    ['TA0008 - Lateral Movement',    'TA0004 - Privilege Escalation'],
    }
    driver_tactics = {
        'phi_log':      'TA0007 - Discovery',
        'phi_cve':      'TA0002 - Execution',
        'phi_risk':     'TA0005 - Defense Evasion',
        'phi_exploit':  'TA0040 - Impact',
        'phi_identity': 'TA0008 - Lateral Movement',
    }
    tactics = list(dict.fromkeys(
        phase_tactics.get(phase, []) + [driver_tactics.get(primary_driver, '')]
    ))
    return [t for t in tactics if t][:4]


def generate_training_pair(idx):
    """Generate one (stage9_input, ideal_report) training pair."""
    rng         = random.Random(idx)
    node_type   = rng.choice(NODE_TYPES)
    provider    = rng.choice(PROVIDERS)
    phase       = rng.choice(ATTACK_PHASES)
    is_malicious= rng.random() > 0.3   # 70% malicious
    node_id     = f"{node_type.lower()}_{idx:04d}"
    scenario_id = f"scenario_{rng.randint(1, 1000):05d}"

    # Threat score correlated with attack phase severity
    phase_score_ranges = {
        'privilege_escalation': (0.75, 0.99),
        'lateral_movement':     (0.65, 0.95),
        'cve_exploitation':     (0.70, 0.99),
        'exfiltration':         (0.60, 0.95),
        'discovery':            (0.40, 0.75),
        'cross_cloud_pivot':    (0.70, 0.99),
    }
    lo, hi      = phase_score_ranges.get(phase, (0.50, 0.90))
    threat_score= round(rng.uniform(lo, hi) if is_malicious else rng.uniform(0.10, 0.50), 4)

    # Severity
    if   threat_score >= 0.90: severity = 'Critical'
    elif threat_score >= 0.75: severity = 'High'
    elif threat_score >= 0.50: severity = 'Medium'
    else:                      severity = 'Low'

    shap         = _rand_shap(threat_score)
    # Bias SHAP toward phase-relevant driver
    phase_driver = {
        'privilege_escalation': 'phi_identity',
        'lateral_movement':     'phi_identity',
        'cve_exploitation':     'phi_cve',
        'exfiltration':         'phi_log',
        'discovery':            'phi_log',
        'cross_cloud_pivot':    'phi_identity',
    }.get(phase, 'phi_log')
    shap[phase_driver] = round(shap[phase_driver] * 2.5, 4)
    total = sum(shap.values())
    shap  = {k: round((v/total)*threat_score, 4) for k, v in shap.items()}

    primary_driver = max(shap, key=shap.get)
    cves           = rng.sample(REAL_CVES, k=rng.randint(0, 3))
    attack_path    = _make_attack_path(node_id, phase)
    mitre_tactics  = _make_mitre(phase, primary_driver)

    # ── Stage 9 input (what the app sends to Mistral) ────────────────────────
    stage9_input = {
        'node_id':        node_id,
        'node_type':      node_type,
        'threat_score':   threat_score,
        'severity':       severity,
        'scenario_id':    scenario_id,
        'shap':           shap,
        'primary_driver': primary_driver,
        'attack_path':    attack_path,
        'mitre_tactics':  mitre_tactics,
        'remediation':    _make_remediation(severity, node_type, cves),
    }

    # ── Ideal report (ground truth label) ───────────────────────────────────
    ideal_report = {
        'node_id':               node_id,
        'severity':              severity,
        'threat_score':          threat_score,
        'summary':               _make_summary(node_id, node_type, severity, phase,
                                               primary_driver, provider, threat_score),
        'attack_narrative':      _make_narrative(node_id, node_type, phase, attack_path,
                                                shap, cves, provider),
        'primary_indicator':     DRIVER_LABELS.get(primary_driver, primary_driver),
        'mitre_tactics':         mitre_tactics,
        'remediation_steps':     _make_remediation(severity, node_type, cves),
        'estimated_blast_radius':_make_blast_radius(node_type, phase, threat_score),
    }

    return stage9_input, ideal_report


# Generate 500 training pairs + 50 validation pairs
N_TRAIN = 500
N_VAL   = 50

train_pairs = [generate_training_pair(i)          for i in range(N_TRAIN)]
val_pairs   = [generate_training_pair(i + N_TRAIN) for i in range(N_VAL)]

print(f'✓ Generated {N_TRAIN} training pairs + {N_VAL} validation pairs')
print()

# Verify distribution
from collections import Counter
sev_dist = Counter(pair[1]['severity'] for pair in train_pairs)
print('Severity distribution:', dict(sev_dist))

# Inspect one sample
s9, rpt = train_pairs[0]
print(f'\nSample input  → severity={s9["severity"]}, threat_score={s9["threat_score"]}')
print(f'Sample summary: {rpt["summary"][:120]}...')

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Tokenize dataset once
def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=1536,
        padding='max_length',
    )

train_tokenized = train_dataset.map(tokenize, batched=True, remove_columns=['text'])
val_tokenized   = val_dataset.map(tokenize,   batched=True, remove_columns=['text'])
train_tokenized.set_format('torch')
val_tokenized.set_format('torch')

# Data collator — standard causal LM (labels = input_ids shifted internally by model)
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir                  = str(OUT_DIR / 'checkpoints'),
    num_train_epochs            = 3,
    per_device_train_batch_size = 2,
    per_device_eval_batch_size  = 2,
    gradient_accumulation_steps = 8,       # effective batch = 16
    warmup_steps                = 50,
    learning_rate               = 2e-4,
    lr_scheduler_type           = 'cosine',
    fp16                        = True,
    logging_steps               = 25,
    eval_strategy               = 'steps',
    eval_steps                  = 100,
    save_strategy               = 'steps',
    save_steps                  = 100,
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    greater_is_better           = False,
    report_to                   = 'none',
    dataloader_pin_memory       = False,
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_tokenized,
    eval_dataset    = val_tokenized,
    data_collator   = collator,
)

print('Starting training...')
print(f'  Steps/epoch : {len(train_tokenized) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}')
print(f'  Total steps : ~{3 * len(train_tokenized) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}')
print(f'  Estimated   : ~3-4 hours on T4')
print()

t0 = time.time()
trainer.train()
print(f'\n✓ Training complete in {(time.time()-t0)/3600:.2f}h')


In [ ]:
# ── CELL 5: LOAD BASE MODEL + TOKENIZER ──────────────────────────────────────
import time
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f'Loading tokenizer from {BASE_MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_ID,
    trust_remote_code=True,
    token=HF_TOKEN,
)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'   # required for causal LM training

# QLoRA quantization config — same as Stage 0b
bnb_config = BitsAndBytesConfig(
    load_in_4bit               = True,
    bnb_4bit_quant_type        = 'nf4',
    bnb_4bit_compute_dtype     = torch.float16,
    bnb_4bit_use_double_quant  = True,
)

print(f'Loading {BASE_MODEL_ID} in 4-bit NF4...')
t0 = time.time()
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config = bnb_config,
    device_map          = 'auto',
    torch_dtype         = torch.float16,
    trust_remote_code   = True,
    token               = HF_TOKEN,
)
print(f'✓ Base model loaded in {(time.time()-t0)/60:.1f} min')
print(f'  Params: {sum(p.numel() for p in base_model.parameters()):,}')

# Required for gradient checkpointing with quantized models
base_model.config.use_cache             = False
base_model.config.pretraining_tp        = 1
base_model.enable_input_require_grads()

In [ ]:
# ── CELL 6: LORA CONFIG ───────────────────────────────────────────────────────
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for QLoRA training
base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r              = 16,          # rank — same as Stage 0b
    lora_alpha     = 32,          # alpha = 2 * rank
    lora_dropout   = 0.05,
    bias           = 'none',
    task_type      = 'CAUSAL_LM',
    # All attention + MLP projection layers — full coverage
    target_modules = [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# Verify trainable param count (should be ~0.7% of total)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'\nTrainable: {trainable:,} / {total:,} = {100*trainable/total:.2f}%')

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Tokenize dataset once
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=1536,
        padding="max_length",
    )

train_tokenized = train_dataset.map(tokenize, batched=True, remove_columns=["text"])
val_tokenized   = val_dataset.map(tokenize,   batched=True, remove_columns=["text"])
train_tokenized.set_format("torch")
val_tokenized.set_format("torch")

# Standard causal LM collator — labels = input_ids, loss ignores padding
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir                  = str(OUT_DIR / "checkpoints"),
    num_train_epochs            = 3,
    per_device_train_batch_size = 2,
    per_device_eval_batch_size  = 2,
    gradient_accumulation_steps = 8,
    warmup_steps                = 50,
    learning_rate               = 2e-4,
    lr_scheduler_type           = "cosine",
    fp16                        = True,
    logging_steps               = 25,
    eval_strategy               = "steps",
    eval_steps                  = 100,
    save_strategy               = "steps",
    save_steps                  = 100,
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    report_to                   = "none",
    dataloader_pin_memory       = False,
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = train_tokenized,
    eval_dataset  = val_tokenized,
    data_collator = collator,
)

print("Starting training...")
steps_per_epoch = len(train_tokenized) // (
    training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
)
print(f"  Steps/epoch : {steps_per_epoch}")
print(f"  Total steps : ~{3 * steps_per_epoch}")
print(f"  Estimated   : ~3-4 hours on T4")
print()

t0 = time.time()
trainer.train()
print(f"\n✓ Training complete in {(time.time()-t0)/3600:.2f}h")


In [ ]:
# ── CELL 8: SAVE ADAPTER LOCALLY ─────────────────────────────────────────────
ADAPTER_DIR = OUT_DIR / 'adapter'
ADAPTER_DIR.mkdir(exist_ok=True)

# Save only the LoRA adapter (not the full 7B model)
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

# Verify saved files
saved = list(ADAPTER_DIR.iterdir())
print('Saved adapter files:')
for f in saved:
    size_mb = f.stat().st_size / 1e6
    print(f'  {f.name}: {size_mb:.2f} MB')

adapter_size = sum(f.stat().st_size for f in saved) / 1e6
print(f'\nTotal adapter size: {adapter_size:.1f} MB (should be ~40-80MB)')

In [ ]:
# ── CELL 9: INFERENCE VALIDATION ─────────────────────────────────────────────
# Test the fine-tuned adapter before pushing to HF.
# Generate reports for 3 held-out examples and verify JSON structure.

from peft import PeftModel
import warnings

print('Loading fine-tuned adapter for validation...')
val_model = model   # already in memory
val_model.eval()


def generate_report(stage9_input: dict, max_new_tokens: int = 512) -> dict:
    prompt   = format_prompt(stage9_input)
    messages = [{'role': 'user', 'content': prompt}]
    text     = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        text, return_tensors='pt',
        truncation=True, max_length=2048,
    ).to(DEVICE)

    with torch.no_grad(), warnings.catch_warnings():
        warnings.simplefilter('ignore')
        out = val_model.generate(
            **inputs,
            max_new_tokens  = max_new_tokens,
            do_sample       = False,
            pad_token_id    = tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        out[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True,
    ).strip()

    # Strip markdown fences
    if '```' in response:
        parts    = response.split('```')
        response = parts[1] if len(parts) > 1 else response
        if response.lstrip().startswith('json'):
            response = response.lstrip()[4:]

    try:
        j0, j1 = response.find('{'), response.rfind('}') + 1
        return json.loads(response[j0:j1])
    except Exception as e:
        return {'_parse_error': str(e), '_raw': response[:300]}


# Validate on 3 held-out validation examples
REQUIRED_FIELDS = [
    'node_id', 'severity', 'threat_score', 'summary',
    'attack_narrative', 'primary_indicator',
    'mitre_tactics', 'remediation_steps', 'estimated_blast_radius'
]

print()
all_pass = True
for i, (s9, expected) in enumerate(val_pairs[:3]):
    print(f'--- Validation {i+1}: {s9["node_id"]} ({s9["severity"]}) ---')
    result = generate_report(s9)

    if '_parse_error' in result:
        print(f'  ✗ JSON parse failed: {result["_parse_error"]}')
        print(f'  Raw: {result["_raw"]}')
        all_pass = False
    else:
        missing = [f for f in REQUIRED_FIELDS if f not in result]
        if missing:
            print(f'  ⚠ Missing fields: {missing}')
        else:
            print(f'  ✓ All required fields present')

        print(f'  severity:   {result.get("severity")} (expected: {expected["severity"]})')
        print(f'  summary:    {str(result.get("summary",""))[:100]}...')
        print(f'  tactics:    {result.get("mitre_tactics", [])}')
        print(f'  steps:      {len(result.get("remediation_steps",[]))} steps')
    print()

print('✓ Validation complete' if all_pass else '⚠ Some validation issues — check above')

In [ ]:
# ── CELL 10: PUSH TO HUGGINGFACE ─────────────────────────────────────────────
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

# Create repo if not exists
try:
    api.create_repo(
        repo_id  = REPO_ID,
        repo_type= 'model',
        exist_ok = True,
        private  = False,
        token    = HF_TOKEN,
    )
    print(f'✓ Repo ready: https://huggingface.co/{REPO_ID}')
except Exception as e:
    print(f'Repo create: {e}')

# Push adapter directory
print(f'Pushing adapter to {REPO_ID}...')
api.upload_folder(
    folder_path = str(ADAPTER_DIR),
    repo_id     = REPO_ID,
    repo_type   = 'model',
    token       = HF_TOKEN,
    commit_message = f'QLoRA adapter: Mistral-7B fine-tuned on {N_TRAIN} Trinetra threat report pairs',
)

# Also push training data for reproducibility
train_data_path = OUT_DIR / 'train_data.json'
with open(train_data_path, 'w') as f:
    json.dump(
        [{'input': s9, 'output': rpt} for s9, rpt in train_pairs],
        f, indent=2
    )
api.upload_file(
    path_or_fileobj = str(train_data_path),
    path_in_repo    = 'train_data.json',
    repo_id         = REPO_ID,
    token           = HF_TOKEN,
    commit_message  = 'Add training data for reproducibility',
)

print()
print('=' * 60)
print('  STAGE 9+10 FINE-TUNING COMPLETE')
print('=' * 60)
print(f'  Adapter:  https://huggingface.co/{REPO_ID}')
print(f'  Base:     {BASE_MODEL_ID}')
print(f'  Pairs:    {N_TRAIN} train / {N_VAL} val')
print(f'  LoRA:     rank=16, alpha=32, 7 target modules')
print()
print('  In app.py load with:')
print(f'  base = AutoModelForCausalLM.from_pretrained("{BASE_MODEL_ID}", ...)')
print(f'  model = PeftModel.from_pretrained(base, "{REPO_ID}")')
print('=' * 60)